# **Agrupación de usuarios con tweets positivos de salud mental**

El objetivo de esta sección es construir un dataset a nivel de usuario, a partir de tweets previamente clasificados como relacionados con riesgo en salud mental (P).

Se transforman datos a nivel tweet → nivel usuario, para poder realizar análisis de perfil.

## **1. Carga de datos**

Se carga el dataset ya anonimizado y clasificado, que contiene tweets con etiquetas de salud mental.

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/mental_health/mental_health_anonymized_clasif.csv",
                         low_memory=False)

Cada fila corresponde a un tweet y el `id` es el identificador único del tweet:

In [ ]:
len(df)

103334

In [ ]:
df.id.nunique()

103334

In [ ]:
df.head()

,id,user_id,userName,name,text,createdAt,description,profile_bio,location,followers,following,retweets,replies,likes,quotes,views,lang,clasif
0,ecf2f84c58900d02,f1d02c3296767781,USR_23f1fc0d,[NOMBRE],5° edición del Taller de Entrenamiento en Auto...,2026-04-16,NaN,Psicóloga experta en Psicología de Emergencias...,<LOCATION>,3025.0,1019.0,0,1,2.0,1.0,128.0,es,C
1,2761732a83ea92be,f1d02c3296767781,USR_23f1fc0d,[NOMBRE],Curso sobre 𝑨𝒕𝒆𝒏𝒄𝒊𝒐𝒏 𝒊𝒏𝒎𝒆𝒅𝒊𝒂𝒕𝒂 𝒂 𝒗𝒊𝒄𝒕𝒊𝒎𝒂𝒔 𝒈𝒊𝒕𝒂...,2026-03-27,NaN,Psicóloga experta en Psicología de Emergencias...,<LOCATION>,3025.0,1019.0,0,0,0.0,0.0,62.0,es,C
2,3d6a8a4df291945a,f1d02c3296767781,USR_23f1fc0d,[NOMBRE],Entrevista en #informacionprevencion dando mi ...,2026-03-26,NaN,Psicóloga experta en Psicología de Emergencias...,<LOCATION>,3025.0,1019.0,0,0,0.0,0.0,37.0,es,C
3,19b036e16d9be62f,f1d02c3296767781,USR_23f1fc0d,[NOMBRE],En mayo tenemos la cita anual de la Comisión d...,2026-03-18,NaN,Psicóloga experta en Psicología de Emergencias...,<LOCATION>,3025.0,1019.0,2,1,2.0,0.0,50.0,es,C
4,0897e1e233902e44,f1d02c3296767781,USR_23f1fc0d,[NOMBRE],V Encuentro Nacional de [USUARIO] \n\nCuidar d...,2026-03-14,NaN,Psicóloga experta en Psicología de Emergencias...,<LOCATION>,3025.0,1019.0,1,0,3.0,0.0,79.0,es,C


## **2. Filtrado de tweets positivos (riesgo)**

Nos quedamos únicamente con los tweets clasificados como **P → posible riesgo en salud mental**

In [ ]:
df_positivo = df[df["clasif"] == "P"].copy()

print(f"{len(df_positivo)} tweets")
print(f"{df_positivo["user_id"].nunique()} usuarios")

2315 tweets
621 usuarios


Esto reduce el dataset a tweets con posible malestar psicológico y usuarios que han expresado al menos un caso de riesgo

## **3. Orden cronológico de los tweets**

Es importante ordenar los tweets para reconstruir la secuencia temporal de cada usuario.

In [ ]:
df_positivo["createdAt"] = pd.to_datetime(df_positivo["createdAt"])

df_positivo = df_positivo.sort_values(
    ["user_id", "createdAt"]
)

## **4. Agrupación de tweets por usuario**

Se concatenan todos los tweets de cada usuario en un único campo de texto.

In [ ]:
df_users = (
    df_positivo
    .groupby("user_id")
    .agg({
        "userName": "first",
        "text": lambda x: "\n\n".join(
            [
                f"[TWEET {i+1}] {tweet}"
                for i, tweet in enumerate(x.astype(str))
            ]
        ),
        "id": "count"
    })
    .reset_index()
)

Renombrado de columnas:

In [ ]:
df_users = df_users.rename(
    columns={
        "text": "tweets_concat",
        "id": "n_tweets"
    }
)

## **5. Resultados**

- Cada fila = 1 usuario

- `tweets_concat` contiene todos sus tweets de riesgo concatenados

- `n_tweets` indica cuántos tweets positivos tiene cada usuario

In [ ]:
df_users.head(10)

,user_id,userName,tweets_concat,n_tweets
0,009c8bb24862d606,USR_0bb8dab2,[TWEET 1] DEP...,1
1,00bcbd0b067ef737,USR_0204aec3,[TWEET 1] RT [USUARIO]: Ya sé que no me lee na...,1
2,00fa95048919fd68,USR_8405c455,[TWEET 1] Stoy faking triste y a punto de puto...,1
3,01fc52c4bc9ffc8c,USR_69f55ce3,[TWEET 1] Cusndo sera el día que yo diga “no m...,10
4,0221573457246aa2,USR_a7b57905,[TWEET 1] Alguien me espía\n\n[TWEET 2] RT [US...,2
5,02502dd835534600,USR_af40ab96,[TWEET 1] de pie por si acaso m atacan,1
6,033933153ad886ca,USR_49120728,[TWEET 1] Te sigo esperando en mi infierno per...,9
7,0482fbaf7bc0eeab,USR_350560c3,[TWEET 1] RT [USUARIO]: Orden de preferencia a...,1
8,0492f3da807b8d30,USR_b477cf15,"[TWEET 1] RT [USUARIO]: te costó levantarte, t...",5
9,04d09936041bc786,USR_22126242,"[TWEET 1] Mi padre tiene un tumor, le operan e...",2


In [ ]:
print(len(df_users))

621


In [ ]:
df_users.n_tweets.sum()

np.int64(2315)

In [ ]:
df_users.to_csv("/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/agrupacion_usuarios_positivos_SM.csv", index=False)

# **Agrupación por usuarios para posterior caracterización**

El objetivo es construir una vista completa del usuario con todos sus tweets y metadatos.

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/mental_health/mental_health_anonymized_clasif.csv",
                         low_memory=False)

df_users = df.copy()

df_users["createdAt"] = pd.to_datetime(df_users["createdAt"])

# Orden cronológico
df_users = df_users.sort_values(
    ["user_id", "createdAt"]
)

In [ ]:
df_users.head()

,id,user_id,userName,name,text,createdAt,description,profile_bio,location,followers,following,retweets,replies,likes,quotes,views,lang,clasif
16464,d4a0bad1127edfbf,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],RT [USUARIO]: Los inmigrantes regularizados po...,2026-04-02,NaN,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,287,62,479.0,30.0,21305.0,es,C
16465,34ea34de2821f795,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],RT [USUARIO]: VOX celebra la sentencia del TSJ...,2026-04-02,NaN,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,145,4,285.0,4.0,8530.0,es,C
16466,07259ea58634ea7c,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],RT [USUARIO]: Una investigación sobre las band...,2026-04-02,NaN,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,17,0,38.0,0.0,697.0,es,C
16467,7558cf36fbeb54a5,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],Otra forma más de robarnos,2026-04-02,NaN,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,0,0,0.0,0.0,0.0,es,C
16468,6345dfffba7e0282,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],RT [USUARIO]: Vecinos de un barrio obrero de <...,2026-04-02,NaN,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,346,13,690.0,5.0,6181.0,es,C


- **Tweets concatenados + estadísticas temporales**

In [ ]:
tweets_grouped = (
    df_users
    .groupby("user_id")
    .agg(
        tweets_concat=(
            "text",
            lambda x: "\n\n".join(
                [
                    f"[TWEET {i+1}] {tweet}"
                    for i, tweet in enumerate(x.astype(str))
                ]
            )
        ),
        n_tweets=("id", "count"),
        first_tweet=("createdAt", "min"),
        last_tweet=("createdAt", "max")
    )
    .reset_index()
)

In [ ]:
tweets_grouped.head()

,user_id,tweets_concat,n_tweets,first_tweet,last_tweet
0,003aa0c7bf89f9e3,[TWEET 1] RT [USUARIO]: Los inmigrantes regula...,93,2026-04-02,2026-04-18
1,009c8bb24862d606,"[TWEET 1] RT [USUARIO]: Esto es mi BCN, noche ...",99,2017-08-17,2026-03-27
2,00bcbd0b067ef737,"[TWEET 1] RT [USUARIO]: ‼️Alto y claro, <PERSO...",98,2026-04-19,2026-04-20
3,00fa95048919fd68,[TWEET 1] i would love to check my phone here\...,67,2024-10-06,2026-04-19
4,011a3468f9d54417,[TWEET 1] Mala Gente.....!!!!! [URL]\n\n[TWEET...,21,2023-01-03,2023-01-12


- **Extracción del último perfil del usuario**

Se selecciona la información más reciente disponible de cada usuario.

In [ ]:
latest_profile = (
    df_users
    .groupby("user_id")
    .tail(1)
    [
        [
            "user_id",
            "userName",
            "name",
            "profile_bio",
            "location",
            "followers",
            "following",
            "lang"
        ]
    ]
)

In [ ]:
latest_profile.head()

,user_id,userName,name,profile_bio,location,followers,following,lang
16415,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,es
59769,009c8bb24862d606,USR_0bb8dab2,[NOMBRE],NaN,Currently in <LOCATION> 🇪🇸,172.0,531.0,es
18159,00bcbd0b067ef737,USR_0204aec3,[NOMBRE],"Viajar, conocer lugares y su gente es mi pasión.","<LOCATION>, <LOCATION>",706.0,672.0,es
63766,00fa95048919fd68,USR_8405c455,[NOMBRE],🔗🔗🔗🔗,en la silla de la tarta :$,125.0,458.0,en
46863,011a3468f9d54417,USR_d78d028e,[NOMBRE],NaN,<PERSON>,1098.0,1128.0,es


- **Merge final (tweets + perfil)**

Se combinan los tweets agregados con los metadatos del usuario.

In [ ]:
df_users_final = tweets_grouped.merge(
    latest_profile,
    on="user_id",
    how="left"
)

- **Reordenación final de columnas**

In [ ]:
df_users_final = df_users_final[
    [
        "user_id",
        "userName",
        "name",
        "tweets_concat",
        "n_tweets",
        "first_tweet",
        "last_tweet",
        "profile_bio",
        "location",
        "followers",
        "following",
        "lang"
    ]
]

In [ ]:
df_users_final.head()

,user_id,userName,name,tweets_concat,n_tweets,first_tweet,last_tweet,profile_bio,location,followers,following,lang
0,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],[TWEET 1] RT [USUARIO]: Los inmigrantes regula...,93,2026-04-02,2026-04-18,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,es
1,009c8bb24862d606,USR_0bb8dab2,[NOMBRE],"[TWEET 1] RT [USUARIO]: Esto es mi BCN, noche ...",99,2017-08-17,2026-03-27,NaN,Currently in <LOCATION> 🇪🇸,172.0,531.0,es
2,00bcbd0b067ef737,USR_0204aec3,[NOMBRE],"[TWEET 1] RT [USUARIO]: ‼️Alto y claro, <PERSO...",98,2026-04-19,2026-04-20,"Viajar, conocer lugares y su gente es mi pasión.","<LOCATION>, <LOCATION>",706.0,672.0,es
3,00fa95048919fd68,USR_8405c455,[NOMBRE],[TWEET 1] i would love to check my phone here\...,67,2024-10-06,2026-04-19,🔗🔗🔗🔗,en la silla de la tarta :$,125.0,458.0,en
4,011a3468f9d54417,USR_d78d028e,[NOMBRE],[TWEET 1] Mala Gente.....!!!!! [URL]\n\n[TWEET...,21,2023-01-03,2023-01-12,NaN,<PERSON>,1098.0,1128.0,es


- **Validación del dataset**

In [ ]:
print(f"Número de usuarios: {len(df_users_final)}")

Número de usuarios: 1124


In [ ]:
print(f"Número total de tweets: {df_users_final.n_tweets.sum()}")

Número total de tweets: 103334


In [ ]:
df_users_final.to_csv(
    "/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/agrupacion_usuarios.csv",
    index=False
)